# Nuclear data uncertainty propagation using the SANDY API

This tutorial shows how to extract, process, and propagate nuclear data uncertainties using the SANDY Python library coupled with NJOY processing tools.

The workflow reflects a complete end‑to‑end path: from raw ENDF‑6 files to covariance sampling, to perturbation of evaluated data.

In [ ]:
import sandy
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Reading and Inspecting ENDF‑6 Data

We begin by retrieving the JEFF‑3.3 evaluated nuclear data file for hydrogen‑1 (ZAM=10010) from the [**IAEA archive**](https://www-nds.iaea.org/public/download-endf/).

Basic [**ENDF-6**](https://www.oecd-nea.org/dbdata/data/manual-endf/endf102.pdf) content inspection (MF=3, MT=1, etc.) allowed us to verify that the cross sections were available and correctly formatted.

> ZAM = Z $\times$ 10000 + A $\times$ 10 + M
> 
> - Z: charge number
> - A: mass number
> - M: metastate (0=ground state, 1=first metastate, ...)
> 
> For H-1, ZAM=10010
> 
> For U-235, ZAM=922350
> 
> For Am-242m, ZAM=952421

In [ ]:
endf = sandy.get_endf6_file("jeff_33", "xs", 10010)

In [ ]:
endf

In [ ]:
print(endf.data[(125, 3, 1)])

In [ ]:
endf.to_file("H1.jeff33")

# Extract cross section data

We reconstructed tabulated **pointwise cross sections** from resonance parameters.

This is done runnning the **RECONR** module of **NJOY**.

The output is a PENDF file, which contains:
 - Linearized cross sections
 - Reconstructed resonance region
 - Continuous energy grid

A 0.1% linearization tolerance (see parameter `err`) is used too speed-up calculations!

In [ ]:
pendf = endf.get_pendf(err=0.1, verbose=True)

Extract the cross sections from the PENDF file and plot them to provide an overview of the $\sigma(E)$ behavior for several MT reactions.

In [ ]:
xs = sandy.Xs.from_endf6(pendf)

In [ ]:
xs.data.plot(logx=True, logy=True)

# Extract covariance data

Covariance data are contained in the ENDF-6 file, but they must be processed to:
- have a unique multi-group structure for all reactions
- reconstruct covariance data when partial contributions are given, e.g., background uncertainty + other uncertainties
- impose constraints if given (total = $\sum$partial)

This is done with **NJOY** (module **ERRORR**) and requires:
- group structure (see parameter `ign`)
- approximate neutron spectrum (see parameter `iwt`)

The output is a ERRORR file.

In [ ]:
errorrs = endf.get_errorr(verbose=True, err=0.1, errorr33_kws=dict(ign=4))

Then, extract the covariance matrix from the ERRORR file.

In [ ]:
cov = errorrs["errorr33"].get_cov()

## Plot the correlation matrix

In [ ]:
sns.heatmap(data=cov.get_corr().data, vmin=-1, vmax=1, cmap="bwr")
plt.axvline(27)
plt.axhline(27)
plt.axvline(27 * 2)
plt.axhline(27 * 2)

## Plot the standard deviation

In [ ]:
std = cov.get_std().reset_index().pivot_table(index="E", columns="MT", values="STD")
std.index = std.index.right
std.plot(drawstyle="steps-pre", logx=True)

# Draw sample of PDF with given covariance matrix

Nuclear data covariance matrices are in relative terms (divide by the mean). Then the PDF is centered in one.

Random variable: $\frac{X}{E[X]}$

Mean value: $E\left[\frac{X}{E[X]}\right]=1$

Variance: $Var\left(\frac{X}{E[X]}\right) = E\left[\left(\frac{X}{E[X]}-1\right)^2\right] = E\left[\left(\frac{X-E[X]}{E[X]}\right)^2\right] = \frac{E\left[\left(X-E[X]\right)^2\right]}{E[X]^2}=\frac{Var(X)}{E[X]^2} \; \xRightarrow{\sqrt{}} \; \frac{\sigma_X}{\mu_X}$

Take a sample of size $N=10000$. Make it reproducible by fixing the seed.

In [ ]:
smp = cov.sampling(10000, seed=1)

## Plot the sample estimate of the correlation matrix

In this step, we compute the sample correlation matrix for the dataset and visualize it using a heatmap.

The matrix `scorr` contains the pairwise correlations between all features in the sample.

In [ ]:
scorr = smp.get_corr()

In [ ]:
sns.heatmap(scorr, vmin=-1, vmax=1, cmap='bwr')
plt.axvline(27)
plt.axhline(27)
plt.axvline(27 * 2)
plt.axhline(27 * 2)

Values range from `-1` (perfect negative correlation) to `+1` (perfect positive correlation).

The colormap `'bwr'` (blue‑white‑red) helps interpret correlations visually:

 - 🔵 Blue → strong negative correlation
 - ⚪ White → no correlation
 - 🔴 Red → strong positive correlation

## Plot the sample estimate of the standard deviation

Compute the **sample standard deviation** for each reaction and visualize how this sample‑based estimate compares to the true or reference standard deviation.

The sample estimate can be computed over different samples sizes by changing this parameter.

In [ ]:
selected_sample_size = 10000

In [ ]:
# plot sample estimate
sstd = smp.data.iloc[:, :selected_sample_size].std(axis=1).rename("STD").reset_index().pivot_table(index="E", columns="MT", values="STD")
sstd.index = sstd.index.right
ax = sstd.add_prefix("SAMPLE EST.: ").plot(drawstyle="steps-pre", logx=True)

# plot reference standard deviation
std.add_prefix("REFERENCE: ").plot(drawstyle="steps-pre", ls="--", logx=True, ax=ax)

# Analyze convergence

When working with a sequence of data points $x_1, x_2, \dots, x_n$, it can be useful to see how statistical measures evolve as more data is added.

This is exactly what the running mean and running standard deviation show.

**Running mean**

The running mean (also called the **cumulative mean**) at position $i$ is the mean of the first $i$ values:

$$
\mathrm{running\;mean}(i) = \frac{1}{i} \sum_{k=1}^i x_k
$$

This tells you:

- how the average changes as new observations come in
- how quickly the average stabilizes
- whether early values dominate or later values pull the mean in a new direction

If the dataset is stable, the running mean will gradually converge to the final mean.

**Running standard deviation**

It measures how spread out the first $i$ data points are, i.e.,

$$
\mathrm{running\;std}(i) = \sqrt{ \frac{1}{i-1} \sum_{k=1}^i \left( x_k - \mathrm{running\;mean(i)} \right)^2 }
$$

This helps you see:

- how the variability changes as more data arrives
- whether the distribution initially appears noisy but stabilizes over time
- how many samples are needed before the standard deviation becomes reliable

Like the running mean, the running standard deviation typically becomes more stable as the sample size increases.

Here we compute and visualize how the running mean and running standard deviation evolve for a selected parameter:

In [ ]:
idx = 0
x = smp.data.iloc[idx]
running_mean = pd.Series({i: np.mean(x[:i]) for i in range(1, len(x)+1)})
running_std = pd.Series({i: np.std(x[:i]) for i in range(1, len(x)+1)})

Both running statistics are then normalized by dividing by their final value.
This allows comparing their relative convergence behavior on the same scale.

This helps illustrate how statistical estimators (mean, std) converge as sample size increases.

In [ ]:
(running_mean / running_mean.iloc[-1]).plot(label="MEAN", logx=True)
(running_std / running_std.iloc[-1]).plot(label="STD", logx=True)
plt.legend()

Running statistics show convergence behavior:

- With few data points: both mean and std fluctuate a lot.
- With more data: they settle toward their “true” values.

Plotting them (especially on a log x‑scale) helps visualize how fast these estimates stabilize.

# Wrap it up in two steps

Create a sample (size $N=20$) from the multi-group relative covariance matrix of $^1$H cross sections. Then store these perturbations in a excel file.

In [ ]:
smps = endf.get_perturbations(20, verbose=True)

Convert the multi-group perturbations into random ENDF-6 files.

In [ ]:
outs = endf.apply_perturbations(smps, to_file=True, verbose=True)